In [ ]:
def wrap_to_pi(a):
    return torch.atan2(torch.sin(a), torch.cos(a))

**Step 1: System Dynamics**

In [ ]:
import torch

def system_dynamics(state_vector, control_input, mass, inertia, g):
    """
    Physics of the drone
    State_vector = [x, y, z, roll, pitch, yaw, vx, vy, vz, w_roll, w_pitch, w_yaw]
    Control_input = [thrust, torque_roll, torque_pitch, torque_yaw]
    """
    # # Physical parameters definition REMEMBER TO ADJUST IF NECESSARY
    # g = 9.81  # gravity [m/s^2]
    # mass = 2.0  # mass of the drone [kg]
    # inertia = torch.tensor([0.0216, 0.0216, 0.04])  # inertia around roll (Ix), pitch (Iy), yaw (Iz) [kg*m^2]
    if not isinstance(inertia, torch.Tensor):
        inertia = torch.tensor(inertia)

    # Unpack state vector
    x, y, z, roll, pitch, yaw, vx, vy, vz, w_roll, w_pitch, w_yaw = state_vector[:,0], state_vector[:,1], state_vector[:,2], state_vector[:,3], state_vector[:,4], state_vector[:,5], state_vector[:,6], state_vector[:,7], state_vector[:,8], state_vector[:,9], state_vector[:,10], state_vector[:,11]
    thrust, torque_roll, torque_pitch, torque_yaw = control_input[:,0], control_input[:,1], control_input[:,2], control_input[:,3]

    # NOTICE THAT: the dataset is recorded in the North-East-Down (NED) frame, so z is positive downwards. The system dynamics then must be consistent with this frame ( ==> vz_dot gravity is positive and thrust is negative)
    # Compute dynamics
    x_dot = vx # world frame
    y_dot = vy # world frame
    z_dot = vz # world frame
    roll_dot = w_roll + (torch.sin(roll) * torch.tan(pitch) * w_pitch) + (torch.cos(roll) * torch.tan(pitch) * w_yaw) # world frame
    pitch_dot = (torch.cos(roll) * w_pitch) - (torch.sin(roll) * w_yaw) # world frame
    yaw_dot = (torch.sin(roll) / torch.cos(pitch) * w_pitch) + (torch.cos(roll) / torch.cos(pitch) * w_yaw) # world frame
    vx_dot = (thrust / mass) * (torch.cos(roll) * torch.sin(pitch) * torch.cos(yaw) + torch.sin(roll) * torch.sin(yaw)) # world frame
    vy_dot = (thrust / mass) * (torch.cos(roll) * torch.sin(pitch) * torch.sin(yaw) - torch.sin(roll) * torch.cos(yaw)) # world frame
    vz_dot = g + (thrust / mass) * (torch.cos(roll) * torch.cos(pitch)) # world frame
    w_roll_dot = (inertia[1]-inertia[2]) / inertia[0] * w_pitch * w_yaw + torque_roll / inertia[0] # body frame
    w_pitch_dot = (inertia[2]-inertia[0]) / inertia[1] * w_roll * w_yaw + torque_pitch / inertia[1] # body frame
    w_yaw_dot = (inertia[0]-inertia[1]) / inertia[2] * w_roll * w_pitch + torque_yaw / inertia[2] # body frame

    state_vector_dot = torch.stack([x_dot, y_dot, z_dot,
                                  roll_dot, pitch_dot, yaw_dot,
                                  vx_dot, vy_dot, vz_dot,
                                  w_roll_dot, w_pitch_dot, w_yaw_dot], dim=1) 

    # Return derivatives of the state vector
    return state_vector_dot

**Step 2: NN model**

In [ ]:
import torch
import torch.nn as nn

class ResidualBModel(nn.Module):
    """
    Residual physics-correcting model.

    Input features (16-dim, normalized internally):
        [sin/cos(roll,pitch,yaw)(6), v_norm(3), w_norm(3), u_norm(4)]
    Output (6-dim, physical units):
        discrete velocity corrections [Δv(3), Δw(3)]

    Velocities (v, w) and controls (u) are normalized using stored statistics
    before being fed to the network.  The output Δv, Δw is in physical units
    (m/s and rad/s) and is added directly to the integrated state.

    dropout_rate: dropout probability applied after each hidden activation
                  (set 0.0 to disable; typical values 0.05 - 0.2)
    """
    def __init__(self, hidden_layers_size, activation_fn, S=None,
                 output_activation=nn.Identity, dropout_rate=0.0,
                 vel_mean=None,     vel_std=None,
                 ang_vel_mean=None, ang_vel_std=None,
                 ctrl_mean=None,    ctrl_std=None):
        super().__init__()
        self.n_out = 6
        n_control = 4
        n_input = 6 + 3 + 3 + n_control  # 16

        layers = [nn.Linear(n_input, hidden_layers_size[0]), activation_fn()]
        if dropout_rate > 0.0:
            layers.append(nn.Dropout(p=dropout_rate))
        for i in range(len(hidden_layers_size) - 1):
            layers += [nn.Linear(hidden_layers_size[i], hidden_layers_size[i + 1]), activation_fn()]
            if dropout_rate > 0.0:
                layers.append(nn.Dropout(p=dropout_rate))
        layers += [nn.Linear(hidden_layers_size[-1], self.n_out), output_activation()]
        self.corr_net = nn.Sequential(*layers)

        for m in self.corr_net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        # Output scaling S: maps raw NN output -> physical velocity units.
        # Default = vel/ang_vel stds so the network learns in normalised space.
        if S is None:
            s = torch.ones(self.n_out, dtype=torch.float32)
        else:
            s = torch.as_tensor(S, dtype=torch.float32).view(-1)
            if s.numel() != self.n_out:
                raise ValueError(f'S must have {self.n_out} elements, got {s.numel()}')
        self.register_buffer('S', s)

        # ---------- normalisation buffers (not trainable) ----------
        def _mbuf(v, size):
            """mean buffer: default zeros"""
            return (torch.zeros(size, dtype=torch.float32) if v is None
                    else torch.as_tensor(v, dtype=torch.float32).view(size))
        def _sbuf(v, size):
            """std buffer: default ones"""
            return (torch.ones(size, dtype=torch.float32) if v is None
                    else torch.as_tensor(v, dtype=torch.float32).view(size))

        self.register_buffer('vel_mean',     _mbuf(vel_mean,     3))
        self.register_buffer('vel_std',      _sbuf(vel_std,      3))
        self.register_buffer('ang_vel_mean', _mbuf(ang_vel_mean, 3))
        self.register_buffer('ang_vel_std',  _sbuf(ang_vel_std,  3))
        self.register_buffer('ctrl_mean',    _mbuf(ctrl_mean,    4))
        self.register_buffer('ctrl_std',     _sbuf(ctrl_std,     4))

    def build_features(self, state_vector, control_input):
        """
        Build normalised 16-dim feature vector.
        state_vector : (B,12) [x,y,z, roll,pitch,yaw, vx,vy,vz, w_roll,w_pitch,w_yaw]
        control_input: (B,4)  [thrust, tau_roll, tau_pitch, tau_yaw]
        """
        roll  = state_vector[:, 3]
        pitch = state_vector[:, 4]
        yaw   = state_vector[:, 5]

        # sin/cos encoding keeps angles in [-1,1] without normalisation
        trig = torch.stack([
            torch.sin(roll), torch.cos(roll),
            torch.sin(pitch), torch.cos(pitch),
            torch.sin(yaw),   torch.cos(yaw),
        ], dim=1)  # (B,6)

        # Normalise linear velocity, angular velocity, and controls
        v = (state_vector[:, 6:9]  - self.vel_mean)     / self.vel_std      # (B,3)
        w = (state_vector[:, 9:12] - self.ang_vel_mean) / self.ang_vel_std  # (B,3)
        u = (control_input          - self.ctrl_mean)    / self.ctrl_std     # (B,4)

        return torch.cat([trig, v, w, u], dim=1)  # (B,16)

    def forward(self, state_vector, control_input):
        z = self.build_features(state_vector, control_input)   # (B,16)
        delta_vw_norm = self.corr_net(z)                       # (B,6) normalised
        delta_vw = delta_vw_norm * self.S                      # (B,6) physical units
        return delta_vw


**Step 3: Data preprocessing**

In [ ]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def dataset_masking(dataset=None):

    # Add dt column at position 1 (limiting the number to 6 decimal points)
    dataset.insert(1, 'dt', dataset['time'].diff().fillna(0).round(6))  # Calculate time intervals (dt) between consecutive samples and place it in a new column 'dt' in position 1

    # Add Euler angles columns (roll, pitch, yaw) at positions 17, 18, 19
    dataset.insert(17, 'roll', 0.0)
    dataset.insert(18, 'pitch', 0.0)
    dataset.insert(19, 'yaw', 0.0)

    # Add the following non-used columns to match the old dataset format
    dataset.insert(24, 'pwm_1', 0.0)
    dataset.insert(25, 'pwm_2', 0.0)
    dataset.insert(26, 'pwm_3', 0.0)
    dataset.insert(27, 'pwm_4', 0.0)
    dataset.insert(28, 'total_thrust', 0.0)

    # Add explicit torque columns (filled by px4_angular_rate_to_torque)
    # Inserted after cmd_yaw_rate so their indices are 33, 34, 35
    dataset.insert(33, 'torque_roll', 0.0)
    dataset.insert(34, 'torque_pitch', 0.0)
    dataset.insert(35, 'torque_yaw', 0.0)

    # Drop the unused columns (from 40 to the end, keeping x_ref..yaw_ref)
    dataset = dataset.drop(columns=dataset.columns[40:])

    return dataset

def from_quaternion_to_euler(dataset=None):
    """
    Convert quaternion orientation to Euler angles (roll, pitch, yaw) in the dataset.
    Quaternion format in dataset: [q_w, q_x, q_y, q_z]
    Euler angles format: [roll, pitch, yaw]
    """
    q_w = dataset[:, 20:21]
    q_x = dataset[:, 21:22]
    q_y = dataset[:, 22:23]
    q_z = dataset[:, 23:24]

    # Compute roll (x-axis rotation)
    sinr_cosp = 2 * (q_w * q_x + q_y * q_z)
    cosr_cosp = 1 - 2 * (q_x**2 + q_y**2)
    roll = np.arctan2(sinr_cosp, cosr_cosp)

    # Compute pitch (y-axis rotation)
    sinp = 2 * (q_w * q_y - q_z * q_x)
    pitch = np.where(np.abs(sinp) >= 1, np.sign(sinp) * (np.pi / 2), np.arcsin(sinp))

    # Compute yaw (z-axis rotation)
    siny_cosp = 2 * (q_w * q_z + q_x * q_y)
    cosy_cosp = 1 - 2 * (q_y**2 + q_z**2)
    yaw = np.arctan2(siny_cosp, cosy_cosp)

    # Replace quaternion columns with Euler angles in the dataset (rounding to 6 decimal points)
    dataset[:, 17:18] = np.round(roll, 6)
    dataset[:, 18:19] = np.round(pitch, 6)
    dataset[:, 19:20] = np.round(yaw, 6)

    return dataset

def px4_pwm_to_thrust(dataset = None, mass=2.0, g=9.81):
    """
    1) I need to clamp the values of cmd_thrust between -1 and 0, because out of this interval the drone will read only -1 and 0, so it doesn't make sense to have values outside this range
    2) The cmd_thrust in the dataset is a PWM (or another signal), so i need to apply a conversion factor to get the actual thrust force. This factor is determined empirically to match the hover condition
    """
    cmd_thrust = dataset[:, 29:30] 
    cmd_thrust = np.clip(cmd_thrust, -1.0, 0.0)  # Clamp between -1 and 0

    thrust = cmd_thrust * mass * g / 0.72 #-0.72 # conversion factor to get thrust in Newtons
    dataset[:, 29:30] = np.round(thrust, 6)

    return dataset



def px4_angular_rate_to_torque(dataset=None, inertia=np.array([0.0216, 0.0216, 0.04])):
    """
    Compute the actual torques applied to the drone by inverting the rotational dynamics equations.
    
    From Euler's rotational equations:
        I_x * alpha_x = (I_y - I_z) * wy * wz + tau_x
        I_y * alpha_y = (I_z - I_x) * wx * wz + tau_y
        I_z * alpha_z = (I_x - I_y) * wx * wy + tau_z
    
    Solving for torques:
        tau_x = I_x * alpha_x - (I_y - I_z) * wy * wz
        tau_y = I_y * alpha_y - (I_z - I_x) * wx * wz
        tau_z = I_z * alpha_z - (I_x - I_y) * wx * wy
    """
    I_x, I_y, I_z = inertia[0], inertia[1], inertia[2]

    # Angular velocities from the dataset
    w_roll = dataset[:, 11:12]
    w_pitch = dataset[:, 12:13]
    w_yaw = dataset[:, 13:14]

    # Angular accelerations from the dataset
    alpha_roll = dataset[:, 14:15]
    alpha_pitch = dataset[:, 15:16]
    alpha_yaw = dataset[:, 16:17]

    # Invert the rotational dynamics to get torques
    tau_x = I_x * alpha_roll - (I_y - I_z) * w_pitch * w_yaw
    tau_y = I_y * alpha_pitch - (I_z - I_x) * w_roll * w_yaw
    tau_z = I_z * alpha_yaw - (I_x - I_y) * w_roll * w_pitch

    # Store torques in the dedicated torque columns (33, 34, 35)
    dataset[:, 33:34] = np.round(tau_x, 6)
    dataset[:, 34:35] = np.round(tau_y, 6)
    dataset[:, 35:36] = np.round(tau_z, 6)

    return dataset

def get_mean_and_std(dataset=None):
    """
    Get mean and standard deviation of the whole dataset for normalization.
    """
    time = dataset[:, 0:1]
    dt = dataset[:, 1:2]
    linear_pos = dataset[:, 2:5] # x, y, z 
    linear_vel = dataset[:, 5:8] # vx, vy, vz
    linear_acc = dataset[:, 8:11] # a_x, a_y, a_z
    angular_vel = dataset[:, 11:14] # w_x, w_y, w_z
    angular_acc = dataset[:, 14:17] # alpha_x, alpha_y, alpha_z
    angular_pos = dataset[:, 17:20] # roll, pitch, yaw
    rest_of_the_data = dataset[:, 20:29] # other data not used for training
    controls = np.hstack([dataset[:, 29:30], dataset[:, 33:36]])  # cmd_thrust, torque_roll, torque_pitch, torque_yaw

    lin_pos_mean = linear_pos.mean(axis=0)
    lin_pos_std = linear_pos.std(axis=0) + 1e-8
    
    lin_vel_mean = linear_vel.mean(axis=0)
    lin_vel_std = linear_vel.std(axis=0) + 1e-8
    
    lin_acc_mean = linear_acc.mean(axis=0)
    lin_acc_std = linear_acc.std(axis=0) + 1e-8
    
    ang_vel_mean = angular_vel.mean(axis=0)
    ang_vel_std = angular_vel.std(axis=0) + 1e-8
    
    ang_acc_mean = angular_acc.mean(axis=0)
    ang_acc_std = angular_acc.std(axis=0) + 1e-8
    
    # rest_mean = rest_of_the_data.mean(axis=0)
    # rest_std = rest_of_the_data.std(axis=0) + 1e-8
    
    controls_mean = controls.mean(axis=0)
    controls_std = controls.std(axis=0) + 1e-8

    mean = np.hstack((lin_pos_mean, lin_vel_mean, ang_vel_mean, lin_acc_mean, ang_acc_mean, controls_mean))
    std = np.hstack((lin_pos_std, lin_vel_std, ang_vel_std, lin_acc_std, ang_acc_std, controls_std))

    return mean, std



def split_data(dataset=None, time_period = 3, dt=0.2, t0 = 25, t1 = 125, t2 = 220):
    """
    Split dataset into training, validation, and testing sets based on time intervals.
    """
    m, _ = dataset.shape

    # Define the delta_t in terms of number of samples
    delta_t = int(time_period / dt)  # number of samples corresponding to the time_period (3/0.2 = 15 samples)

    # Find the first index that corresponding to t0, t1, t2
    start_t0 = np.searchsorted(dataset[:, 0], t0) # index where time >= t0 ( it takes the first index that satisfies the condition)
    start_t1 = np.searchsorted(dataset[:, 0], t1) 
    start_t2 = np.searchsorted(dataset[:, 0], t2)

    end_t0 = start_t0 + delta_t
    end_t1 = start_t1 + delta_t
    end_t2 = start_t2 + delta_t 
    
    data_test_0 = dataset[ start_t0:end_t0, : ]  
    data_test_1 = dataset[ start_t1:end_t1, : ]  
    data_test_2 = dataset[ start_t2:end_t2, : ]

    # Make an array for the tests
    data_test = [data_test_0, data_test_1, data_test_2]

    # Separate testing data from the rest
    data_train_val_0 = dataset[0:start_t0, :]
    data_train_val_1 = dataset[end_t0:start_t1, :]
    data_train_val_2 = dataset[end_t1:start_t2, :]
    data_train_val_3 = dataset[end_t2:m, :]

    # Ensure that the testing/validation samples are evenly distributed in each set, to ease the pair creation (current, next)
    if (data_train_val_0.shape[0] % 2) != 0:
        data_train_val_0 = data_train_val_0[:-1, :] # remove the last sample if odd
    if (data_train_val_1.shape[0] % 2) != 0:
        data_train_val_1 = data_train_val_1[:-1, :]
    if (data_train_val_2.shape[0] % 2) != 0:
        data_train_val_2 = data_train_val_2[:-1, :]
    if (data_train_val_3.shape[0] % 2) != 0:
        data_train_val_3 = data_train_val_3[:-1, :]

    data_train_val = np.vstack((data_train_val_0, data_train_val_1, data_train_val_2, data_train_val_3))

    return data_test, data_train_val
    
def configure_data(dataset):
    m, n = dataset.shape
    dt_test = dataset[:, 1:2]
    linear_pos = dataset[:,2:5]
    linear_vel = dataset[:,5:8]
    linear_acc = dataset[:, 8:11]
    angular_vel = dataset[:,11:14]
    angular_acc = dataset[:, 14:17]
    angular_pos = dataset[:,17:20]
    states = np.hstack((linear_pos, angular_pos, linear_vel, angular_vel, linear_acc, angular_acc))
    controls = np.hstack([dataset[:, 29:30], dataset[:, 33:36]])  # cmd_thrust, torque_roll, torque_pitch, torque_yaw

    # states = torch.tensor(np.array(states, dtype=np.float32))
    # controls = torch.tensor(np.array(controls, dtype=np.float32))
    # dt_test = torch.tensor(np.array(dt_test, dtype=np.float32))

    return states, controls, dt_test

def create_and_shuffle_pairs(data=None):
    """
    Create non-overlapping (current, next) pairs and shuffle them.
    This ensures each state appears exactly once in the dataset.
    """
    n_pairs = (data.shape[0] - 1)  # All possible consecutive pairs
    
    # Stack current and next states
    current_samples = data[:-1]   # All except last
    next_samples = data[1:]       # All except first
    
    # Create indices and shuffle
    indices = np.arange(n_pairs)
    np.random.seed(42)
    np.random.shuffle(indices)
    
    # Return shuffled pairs
    return current_samples[indices], next_samples[indices] 

def configure_training_and_validation_data(data_current=None, data_next=None):
    """
    Prepare paired data for training. 
    
    Args:
        data_current: Array of current states
        data_next: Array of corresponding next states
    """
    # Extract current state features
    linear_pos_curr = data_current[:,2:5]
    linear_vel_curr = data_current[:,5:8]
    linear_acc_curr = data_current[:, 8:11]
    angular_vel_curr = data_current[:,11:14]
    angular_acc = data_current[:, 14:17]
    angular_pos_curr = data_current[:,17:20]
    states_curr = np.hstack((linear_pos_curr, angular_pos_curr, linear_vel_curr, angular_vel_curr, linear_acc_curr, angular_acc))
    controls_curr = np.hstack([data_current[:, 29:30], data_current[:, 33:36]])  # cmd_thrust, torque_roll, torque_pitch, torque_yaw
    
    # Extract next state features
    dt = data_next[: ,1]  # time step between current and next state
    linear_pos_next = data_next[:,2:5]
    linear_vel_next = data_next[:,5:8]
    linear_acc_next = data_next[:, 8:11]
    angular_vel_next = data_next[:,11:14]
    angular_acc_next = data_next[:, 14:17]
    angular_pos_next = data_next[:,17:20]
    states_next = np.hstack((linear_pos_next, angular_pos_next, linear_vel_next, angular_vel_next, linear_acc_next, angular_acc_next))
    controls_next = np.hstack([data_next[:, 29:30], data_next[:, 33:36]])  # cmd_thrust, torque_roll, torque_pitch, torque_yaw

    states_curr = np.array(states_curr, dtype=np.float32)
    states_next = np.array(states_next, dtype=np.float32)
    controls_curr = np.array(controls_curr, dtype=np.float32)
    controls_next = np.array(controls_next, dtype=np.float32)
    dt = np.array(dt, dtype=np.float32)
    
    # vel_curr_norm, vel_next_norm, controls_norm = normalize_data(states_curr[:,3:], states_next[:,3:], controls) # to normalize only velocities and not positions

    # states_curr_norm = np.hstack((states_curr[:, :3], vel_curr_norm))
    # states_next_norm = np.hstack((states_next[:, :3], vel_next_norm))
    
    # Convert to tensors
    X_curr = torch.tensor(states_curr)
    X_next = torch.tensor(states_next)
    U_curr = torch.tensor(controls_curr)
    U_next = torch.tensor(controls_next)
    dt = torch.tensor(dt)
    
    return X_curr, X_next, U_curr, U_next, dt


def save_pairs_to_csv(X_current=None, X_next=None, U_curr=None, U_next=None, dt=None, filename=None):
    """
    Save paired data to CSV with clear structure for inspection.
    """
    import pandas as pd
    
    # Convert tensors to numpy if needed
    if isinstance(X_current, torch. Tensor):
        X_current = X_current.cpu().numpy()
        X_next = X_next. cpu().numpy()
        U_curr = U_curr.cpu().numpy()
        U_next = U_next.cpu().numpy()
        dt = dt.cpu().numpy()
    
    # Create a dataframe with descriptive column names
    df = pd.DataFrame({
        # Pair index
        'pair_idx':  np.arange(len(X_current)),
        
        # Time step
        'dt': dt,
        
        # Current state
        'curr_x': X_current[:, 0],
        'curr_y': X_current[:, 1],
        'curr_z': X_current[:, 2],
        'curr_roll': X_current[:, 3],
        'curr_pitch': X_current[:, 4],
        'curr_yaw': X_current[:, 5],
        'curr_vx': X_current[:, 6],
        'curr_vy': X_current[:, 7],
        'curr_vz': X_current[:, 8],
        'curr_wx': X_current[:, 9],
        'curr_wy': X_current[:, 10],
        'curr_wz': X_current[:, 11],
        'curr_ax': X_current[:, 12],
        'curr_ay': X_current[:, 13],
        'curr_az': X_current[:, 14],
        'curr_alpha_x': X_current[:, 15],
        'curr_alpha_y': X_current[:, 16],
        'curr_alpha_z': X_current[:, 17],

        # Current controls
        'thrust': U_curr[:, 0],
        'torque_roll': U_curr[:, 1],
        'torque_pitch': U_curr[:, 2],
        'torque_yaw': U_curr[: , 3],
        
        # Next state
        'next_x': X_next[:, 0],
        'next_y': X_next[:, 1],
        'next_z': X_next[:, 2],
        'next_roll': X_next[:, 3],
        'next_pitch': X_next[:, 4],
        'next_yaw': X_next[:, 5],
        'next_vx': X_next[:, 6],
        'next_vy': X_next[:, 7],
        'next_vz': X_next[:, 8],
        'next_wx': X_next[:, 9],
        'next_wy': X_next[:, 10],
        'next_wz': X_next[:, 11],
        'next_ax': X_next[:, 12],
        'next_ay': X_next[:, 13],
        'next_az': X_next[:, 14],
        'next_alpha_x': X_next[:, 15],
        'next_alpha_y': X_next[:, 16],
        'next_alpha_z': X_next[:, 17],

        # Next controls
        'thrust': U_next[:, 0],
        'torque_roll': U_next[:, 1],
        'torque_pitch': U_next[:, 2],
        'torque_yaw': U_next[: , 3],
    })
    
    
    # Save to CSV
    df.to_csv(filename, index=False, float_format='%.6f')
    print(f"✅ Saved {len(df)} pairs to '{filename}'")
    
    return df



def normalize_NN_inputs(X_current=None, U_curr=None, mean=None, std=None):
    """
    Normalize NN inputs using provided mean and std.
    """
    # Unpack mean and std
    lin_pos_mean, lin_vel_mean, ang_vel_mean, lin_acc_mean, ang_acc_mean, controls_mean = mean[:3], mean[3:6], mean[6:9], mean[9:12], mean[12:15], mean[15:19]
    lin_pos_std, lin_vel_std, ang_vel_std, lin_acc_std, ang_acc_std, controls_std = std[:3], std[3:6], std[6:9], std[9:12], std[12:15], std[15:19]

    # Normalize current states
    linear_pos_curr_norm = (X_current[:3] - lin_pos_mean) / lin_pos_std
    linear_vel_curr_norm = (X_current[6:9] - lin_vel_mean) / lin_vel_std
    angular_vel_curr_norm = (X_current[9:12] - ang_vel_mean) / ang_vel_std
    # linear_acc_curr_norm = (X_current[12:15] - lin_acc_mean) / lin_acc_std
    # angular_acc_curr_norm = (X_current[15:18] - ang_acc_mean) / ang_acc_std
    controls_curr_norm = (U_curr - controls_mean) / controls_std

    # Reconstruct normalized current state tensor
    X_current_norm = np.concatenate((linear_pos_curr_norm, X_current[3:6], linear_vel_curr_norm, angular_vel_curr_norm)) #, linear_acc_curr_norm, angular_acc_curr_norm))
    U_curr_norm = controls_curr_norm

    return X_current_norm, U_curr_norm



def denormalize_NN_outputs(X_pred=None, mean=None, std=None):
    """
    Denormalize NN outputs using provided mean and std.
    """
    # Move mean and std from cpu to the same device as X_pred
    mean = torch.tensor(mean, dtype=torch.float32, device=X_pred.device)
    std = torch.tensor(std, dtype=torch.float32, device=X_pred.device)

    # Unpack mean and std
    lin_pos_mean, lin_vel_mean, ang_vel_mean, lin_acc_mean, ang_acc_mean, controls_mean = mean[:3], mean[3:6], mean[6:9], mean[9:12], mean[12:15], mean[15:19]
    lin_pos_std, lin_vel_std, ang_vel_std, lin_acc_std, ang_acc_std, controls_std = std[:3], std[3:6], std[6:9], std[9:12], std[12:15], std[15:19]

    # Denormalize predicted states
    linear_acc_pred_denorm = X_pred[:,:3] * lin_acc_std + lin_acc_mean
    angular_acc_pred_denorm = X_pred[:,3:6] * ang_acc_std + ang_acc_mean
    linear_vel_pred_denorm = X_pred[:,6:9] * lin_vel_std + lin_vel_mean
    angular_vel_pred_denorm = X_pred[:,9:12] * ang_vel_std + ang_vel_mean
    

    # Reconstruct denormalized next state tensor
    # X_pred_denorm = np.concatenate((linear_acc_denorm, angular_acc_denorm, linear_vel_denorm, angular_vel_denorm))
    X_pred_norm = torch.cat((linear_acc_pred_denorm, angular_acc_pred_denorm, linear_vel_pred_denorm, angular_vel_pred_denorm), dim=1) # dim=1 to concatenate along the feature dimension

    return X_pred_norm

def diff_sysdyn_dataset(state_vector, control_input, mass, inertia, g, dt, X_next):
    """
    Compute the error between the approximated system dynamics and the dataset derivatives.
    """
    # Compute the approximated derivatives using the system dynamics function
    state_vector_dot_approx = system_dynamics(state_vector, control_input, mass, inertia, g)

    # Euler integration to get the next state from the current state and the approximated derivatives
    state_vector_next_approx = state_vector + state_vector_dot_approx * dt

    # Compute the error between the approximated and actual next states
    error = X_next - state_vector_next_approx

    return error, state_vector_next_approx


In [ ]:
import pickle

import pandas as pd
import numpy as np
import yaml

# from c_Data_preprocessing import *

#import numpy as np

# 1) import the physical parameters
with open("dataset/physical_param_config.yaml", 'r') as f: # Open the YAML file
    physical_params = yaml.safe_load(f) # Load the content into a Python dictionary
    mass, inertia, g = physical_params["Physical_parameters"]["mass"], physical_params["Physical_parameters"]["inertia"], physical_params["Physical_parameters"]["g"]
    thrust_min, torque_roll_min, torque_pitch_min, torque_yaw_min = physical_params["Physical_parameters"]["thrust_min"], physical_params["Physical_parameters"]["torque_roll_min"], physical_params["Physical_parameters"]["torque_pitch_min"], physical_params["Physical_parameters"]["torque_yaw_min"]
    thrust_max, torque_roll_max, torque_pitch_max, torque_yaw_max = physical_params["Physical_parameters"]["thrust_max"], physical_params["Physical_parameters"]["torque_roll_max"], physical_params["Physical_parameters"]["torque_pitch_max"], physical_params["Physical_parameters"]["torque_yaw_max"]
    min_cmd_values = [thrust_min, torque_roll_min, torque_pitch_min, torque_yaw_min]
    max_cmd_values = [thrust_max, torque_roll_max, torque_pitch_max, torque_yaw_max]

   
# 2) Load dataset
dataset_rough = pd.read_csv('dataset/data_set_drone.csv')
# Print the type of each column
#print(f"data_set_drone columns types: {dataset.dtypes}")

# 2a) I want to lead back the new dataset to the previous format (of the old dataset in order to not modify the rest of the code)
dataset_masked = dataset_masking(dataset_rough)

# 2b) Save the new dataset
dataset_masked.to_csv('dataset/dataset_masked.csv', index=False)
#print(f"Cmd_thrust: min={dataset['cmd_thrust'].min()}, max={dataset['cmd_thrust'].max()}")



# 3) Preprocess of the dataset
# 3a) Complete the dataset with the missing information
dataset_original = np.array(dataset_masked)
dataset_original = from_quaternion_to_euler(dataset_original)
dataset_original = px4_pwm_to_thrust(dataset = dataset_original, mass=mass, g=g)  # Convert PX4 signals (cmd_thrust (probably [PWM])) to cmd_thrust values [N]
dataset_original = px4_angular_rate_to_torque(dataset= dataset_original, inertia = inertia)  # Convert PX4 angular rates (cmd_bodyrates (probably [rad/s])) to torques [N*m]


# 3b) Generate a new dataset and merge it with the original one
# dataset_generated = generate_additional_dataset(dataset=dataset_original,min_cmd_values=min_cmd_values, max_cmd_values=max_cmd_values, mass= mass, inertia=inertia, g=g)


# 3c) Add artificial noise to the generated dataset to make it more realistic and improve the NN generalization
# dataset_generated = add_noise_to_dataset(dataset_generated) # Adjust the noise standard deviation as needed


# 3d) Preprocess the generated dataset
# dataset_generated = round_dataset(dataset_generated, decimals=6) 
# dataset_generated = remove_first_row(dataset_generated) 


# 3e) Merge the original dataset with the generated one
# dataset = np.vstack((dataset_original, dataset_generated)) # Merge the original dataset with the generated one


# 3f) Save the preprocessed datasets
pd.DataFrame(dataset_original).to_csv('dataset/dataset_original.csv', index=False)
# pd.DataFrame(dataset_generated).to_csv('dataset/dataset_generated.csv', index=False)
# pd.DataFrame(dataset).to_csv('dataset/dataset.csv', index=False)

**Step 4: Dataset visualization**

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load dataset
DATASET_PATH = os.path.join("dataset", "dataset_original.csv")
dataset_df   = pd.read_csv(DATASET_PATH)
dataset_np   = dataset_df.to_numpy(dtype=np.float32)

# dataset_original.csv is already fully preprocessed by the cell above
# (px4_pwm_to_thrust and px4_angular_rate_to_torque have already been applied).
# Only from_quaternion_to_euler is re-applied here since it is idempotent
# (reads quaternion columns 20:24, writes Euler angles 17:20).
dataset_np = from_quaternion_to_euler(dataset_np)

# ----- Global normalisation statistics -----
# mean / std layout:
#   [0:3]  lin_pos   [3:6]  lin_vel   [6:9]  ang_vel
#   [9:12] lin_acc  [12:15] ang_acc  [15:19] controls (4)
mean, std = get_mean_and_std(dataset_np)

# Extract stats used for NN input normalisation
vel_mean     = mean[3:6];   vel_std     = std[3:6]
ang_vel_mean = mean[6:9];   ang_vel_std = std[6:9]
ctrl_mean    = mean[15:19]; ctrl_std    = std[15:19]

# ----- Train / val / test split -----
# time_period=1  ->  each test segment covers exactly 1 second of flight
dt_candidates = dataset_np[:, 1]
dt_nominal    = float(np.median(dt_candidates[dt_candidates > 0])) if np.any(dt_candidates > 0) else 0.01
data_test_segments, data_train_val = split_data(
    dataset_np, time_period=1, dt=dt_nominal, t0=25, t1=125, t2=220)

# Shuffle and split train / val
pairs_curr, pairs_next = create_and_shuffle_pairs(data_train_val)
split_idx = int(0.8 * len(pairs_curr))
split_idx = min(max(split_idx, 1), len(pairs_curr) - 1)

train_data_current, val_data_current = pairs_curr[:split_idx], pairs_curr[split_idx:]
train_data_next,    val_data_next    = pairs_next[:split_idx], pairs_next[split_idx:]

X_train, X_train_next, U_train, U_train_next, dt_train = configure_training_and_validation_data(
    train_data_current, train_data_next)
X_val,   X_val_next,   U_val,   U_val_next,   dt_val   = configure_training_and_validation_data(
    val_data_current, val_data_next)

# Keep 12-state slice used by physics / loss
X_train_phys      = X_train[:, :12].float()
X_train_next_phys = X_train_next[:, :12].float()
X_val_phys        = X_val[:, :12].float()
X_val_next_phys   = X_val_next[:, :12].float()
U_train = U_train.float(); dt_train = dt_train.float()
U_val   = U_val.float();   dt_val   = dt_val.float()

# ----- Quick dataset overview -----
time = dataset_np[:, 0]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(time, dataset_np[:, 2], label="x")
ax[0].plot(time, dataset_np[:, 3], label="y")
ax[0].plot(time, dataset_np[:, 4], label="z")
ax[0].set_title("Linear position"); ax[0].set_xlabel("Time [s]")
ax[0].set_ylabel("Position [m]"); ax[0].legend(); ax[0].grid(True)
ax[1].plot(time, dataset_np[:, 5], label="vx")
ax[1].plot(time, dataset_np[:, 6], label="vy")
ax[1].plot(time, dataset_np[:, 7], label="vz")
ax[1].set_title("Linear velocity"); ax[1].set_xlabel("Time [s]")
ax[1].set_ylabel("Velocity [m/s]"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

print(f"Dataset: {DATASET_PATH}  |  {len(dataset_np)} samples  |  dt_nominal={dt_nominal:.4f} s")
print(f"Train pairs: {len(train_data_current)}  |  Val pairs: {len(val_data_current)}")
print(f"Test segments (1 s each): {[seg.shape[0] for seg in data_test_segments]}")
print(f"vel_mean={vel_mean}, vel_std={vel_std}")
print(f"ctrl_mean={ctrl_mean}, ctrl_std={ctrl_std}")


**Step 5: Data Loss**

In [ ]:
import torch
import torch.nn as nn
# from a_System_dynamics.system_dynamics import system_dynamics

_mse_loss = nn.MSELoss()

def compute_new_pos_angles(delta_vw, X_curr, U_curr, dt, mass, inertia, g):
    if dt.ndim == 1:
        dt = dt.view(-1, 1)

    # baseline physics derivative
    x_dot_phys = system_dynamics(X_curr, U_curr, mass, inertia=inertia, g=g)  # (B,12)

    # NN correction (delta_vw)
    delta_v = delta_vw[:, 0:3]  # (B,3) correction for linear velocities
    delta_w = delta_vw[:, 3:6]  # (B,3) correction for angular velocities

    x_next_base = X_curr + x_dot_phys * dt  # Next state

    v_next = x_next_base[:, 6:9] + delta_v  # New predicted next linear velocity with correction
    w_next = x_next_base[:, 9:12] + delta_w  # New predicted next angular velocity with correction

    # 3) integrate pose using corrected velocities/angular velocities
    roll  = X_curr[:, 3]
    pitch = X_curr[:, 4]
    yaw   = X_curr[:, 5]

    vx, vy, vz = v_next[:, 0], v_next[:, 1], v_next[:, 2]
    w_roll, w_pitch, w_yaw = w_next[:, 0], w_next[:, 1], w_next[:, 2]

    # position derivatives in world frame
    x_dot = vx
    y_dot = vy
    z_dot = vz

    # Euler angle rates using corrected angular rates
    # NOTE: this has singularities at cos(pitch)=0 (pitch = +- 90 deg)
    roll_dot  = w_roll + torch.sin(roll) * torch.tan(pitch) * w_pitch + torch.cos(roll) * torch.tan(pitch) * w_yaw
    pitch_dot = torch.cos(roll) * w_pitch - torch.sin(roll) * w_yaw
    yaw_dot   = (torch.sin(roll) / torch.cos(pitch)) * w_pitch + (torch.cos(roll) / torch.cos(pitch)) * w_yaw

    pos_angles_dot = torch.stack([x_dot, y_dot, z_dot, roll_dot, pitch_dot, yaw_dot], dim=1)  # (B,6)
    pos_angles_next = X_curr[:, 0:6] + pos_angles_dot * dt                                    # (B,6) Integrate to get new position and angles

    # Assemble final corrected next state
    X_next = x_next_base.clone()
    X_next[:, 0:6] = pos_angles_next
    X_next[:, 6:9] = v_next
    X_next[:, 9:12] = w_next

    return X_next


def data_loss(model, X_curr, U_curr_NN, X_next, dt,
              mass=2.0, inertia=torch.tensor([0.0217, 0.0217, 0.04]), g=9.81,
              channel_weights=None, lambda_corr=0.0):
    """
    channel_weights: (12,) tensor weighting each state channel.
                     Positions/angles (0-5) use smaller weights than velocities (6-11).
    lambda_corr:     L2 penalty on the NN correction magnitude (keeps corrections small,
                     ensures the physics baseline is not fully overridden).
    """
    if dt.ndim == 1:
        dt = dt.view(-1, 1)

    delta_vw = model(X_curr, U_curr_NN)  # (B,6)

    X_next_NN = compute_new_pos_angles(delta_vw, X_curr, U_curr_NN, dt, mass, inertia, g)  # (B,12)

    # channel-wise MSE over all 12 state channels
    per_channel_losses = torch.mean((X_next_NN - X_next) ** 2, dim=0)  # (12,)

    if channel_weights is None:
        # Default: velocity/rate channels weighted 1.0, position/angle channels weighted 0.1
        cw = torch.cat([
            torch.full((6,), 0.1, device=X_curr.device, dtype=X_curr.dtype),  # pos + angles
            torch.ones(6,         device=X_curr.device, dtype=X_curr.dtype),   # vel + rates
        ])
    else:
        cw = channel_weights.to(device=X_curr.device, dtype=X_curr.dtype)

    loss_data = torch.sum(cw * per_channel_losses)

    # L2 penalty: discourage the NN from producing large corrections that override physics
    loss_corr = torch.mean(delta_vw ** 2)

    total_loss = loss_data + lambda_corr * loss_corr
    return total_loss, {
        "loss_data": loss_data.detach(),
        "loss_corr": loss_corr.detach(),
        "per_channel": per_channel_losses.detach(),
    }


**Step 6: Training and Validation**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Hyper-parameters ──────────────────────────────────────────
HIDDEN_LAYERS  = [64, 64, 32]   # smaller model → better generalisation
DROPOUT_RATE   = 0.1            # dropout between hidden layers
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-4           # L2 weight regularisation (Adam)
BATCH_SIZE     = 512            # mini-batch size for stochastic gradient noise
NUM_EPOCHS     = 50000
GRAD_CLIP      = 1.0            # max gradient norm
LAMBDA_CORR    = 1e-4           # penalty on large NN corrections (keeps physics dominant)
EARLY_STOP_PAT = 3000           # epochs without val-loss improvement before stopping
MIN_DELTA      = 1e-7           # minimum improvement to reset early-stopping counter

# Build output scaling S = [vel_std(3), ang_vel_std(3)] so the network
# learns corrections in normalised space and S de-normalises the output
# back to physical velocity units (m/s and rad/s).
S_scale = np.concatenate([vel_std, ang_vel_std]).astype(np.float32)

# ── Model ───────────────────────────────────────────────
model = ResidualBModel(
    hidden_layers_size=HIDDEN_LAYERS,
    activation_fn=nn.ELU,
    output_activation=nn.Identity,
    dropout_rate=DROPOUT_RATE,
    S=S_scale,
    vel_mean=vel_mean,         vel_std=vel_std,
    ang_vel_mean=ang_vel_mean, ang_vel_std=ang_vel_std,
    ctrl_mean=ctrl_mean,       ctrl_std=ctrl_std,
).to(device)

# ── Optimizer (with L2 weight decay) ───────────────────────
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ── LR schedulers (cosine warm-down then plateau reduction) ────────
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=3500, eta_min=1e-4)
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=300,
    threshold=1e-5, threshold_mode='abs', min_lr=1e-5)

# ── Move data to device ───────────────────────────────────
X_train_phys_d      = X_train_phys.to(device)
X_train_next_phys_d = X_train_next_phys.to(device)
U_train_d           = U_train.to(device)
dt_train_d          = dt_train.to(device)

X_val_phys_d      = X_val_phys.to(device)
X_val_next_phys_d = X_val_next_phys.to(device)
U_val_d           = U_val.to(device)
dt_val_d          = dt_val.to(device)

# ── Channel weights (12 channels: pos/angles small, velocities full) ───
# Positions (0-2) and Euler angles (3-5) get a small weight so the loss
# focuses on what the NN directly corrects (velocities 6-11), while still
# penalising large position/angle drift.
channel_weights = torch.tensor(
    [0.1, 0.1, 0.1,   # x, y, z
     0.1, 0.1, 0.1,   # roll, pitch, yaw
     1.0, 1.0, 1.0,   # vx, vy, vz
     1.0, 1.0, 1.0],  # w_roll, w_pitch, w_yaw
    device=device, dtype=X_train_phys_d.dtype)

# ── Mini-batch DataLoader ───────────────────────────────────
train_dataset = TensorDataset(X_train_phys_d, U_train_d,
                               X_train_next_phys_d, dt_train_d)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ── Training state ──────────────────────────────────────────
train_losses   = []
val_losses     = []
lr_history     = []
per_ch_train   = [[] for _ in range(6)]   # per-channel MSE (channels 6-11: vx…w_yaw)
per_ch_val     = [[] for _ in range(6)]

best_val_loss    = float('inf')
best_model_state = None
patience_counter = 0

mass_    = 2.0
inertia_ = torch.tensor([0.0217, 0.0217, 0.04], device=device)
g_       = 9.81

# ── Training loop ──────────────────────────────────────────
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_train_loss = 0.0
    epoch_ch_train   = torch.zeros(6, device=device)

    for X_b, U_b, Xn_b, dt_b in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss_b, extras_b = data_loss(
            model=model,
            X_curr=X_b, U_curr_NN=U_b, X_next=Xn_b, dt=dt_b,
            mass=mass_, inertia=inertia_, g=g_,
            channel_weights=channel_weights,
            lambda_corr=LAMBDA_CORR,
        )
        loss_b.backward()
        # Gradient clipping – prevents exploding gradients with ELU nets
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        epoch_train_loss += loss_b.item() * X_b.size(0)
        epoch_ch_train   += extras_b['per_channel'][6:12] * X_b.size(0)

    epoch_train_loss /= len(train_dataset)
    epoch_ch_train   = (epoch_ch_train / len(train_dataset)).cpu()

    # ── Validation (full batch, no gradient) ───────────────────
    model.eval()
    with torch.no_grad():
        val_loss_t, extras_val = data_loss(
            model=model,
            X_curr=X_val_phys_d, U_curr_NN=U_val_d,
            X_next=X_val_next_phys_d, dt=dt_val_d,
            mass=mass_, inertia=inertia_, g=g_,
            channel_weights=channel_weights,
            lambda_corr=LAMBDA_CORR,
        )
    val_loss  = float(val_loss_t.item())
    val_ch    = extras_val['per_channel'][6:12].cpu()

    # ── LR schedule ────────────────────────────────────────
    if optimizer.param_groups[0]['lr'] > 1e-4:
        scheduler_cosine.step()
    else:
        scheduler_plateau.step(val_loss)

    train_losses.append(epoch_train_loss)
    val_losses.append(val_loss)
    lr_history.append(optimizer.param_groups[0]['lr'])
    for ch in range(6):
        per_ch_train[ch].append(epoch_ch_train[ch].item())
        per_ch_val[ch].append(val_ch[ch].item())

    # ── Early stopping ───────────────────────────────────────
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss    = val_loss
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PAT:
            print(f'Early stopping at epoch {epoch + 1}  '
                  f'(no improvement for {EARLY_STOP_PAT} epochs)')
            break

    if (epoch + 1) % 500 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:05d}/{NUM_EPOCHS} | train: {epoch_train_loss:.6f} '
              f'| val: {val_loss:.6f} | best_val: {best_val_loss:.6f} | lr: {lr:.2e}')

# ── Restore best checkpoint ────────────────────────────────
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'Restored best model (val loss: {best_val_loss:.6f})')
print('Training complete.')

# ── Enhanced training-curve plot ───────────────────────────────────
# overfitting plot uses the np already imported above
_epochs   = np.arange(1, len(train_losses) + 1)
_train_np = np.array(train_losses)
_val_np   = np.array(val_losses)
_ratio    = _val_np / (_train_np + 1e-12)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1 – linear scale
axes[0].plot(_epochs, _train_np, label='Train', lw=1.2)
axes[0].plot(_epochs, _val_np,   label='Val',   lw=1.2)
axes[0].set_title('Loss – linear scale')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2 – log scale (makes early/late improvement visible)
axes[1].semilogy(_epochs, _train_np, label='Train', lw=1.2)
axes[1].semilogy(_epochs, _val_np,   label='Val',   lw=1.2)
# Shade epochs where val > 1.5 × train (overfitting zone)
_overfit_mask = _ratio > 1.5
if _overfit_mask.any():
    axes[1].fill_between(_epochs, _val_np.min() * 0.9, _val_np,
                         where=_overfit_mask, alpha=0.25, color='red',
                         label='val/train > 1.5')
axes[1].set_title('Loss – log scale')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, which='both')

# Panel 3 – overfitting ratio
axes[2].plot(_epochs, _ratio, color='darkorange', lw=1.2, label='val / train')
axes[2].axhline(1.0, color='k',   ls='--', lw=0.9, label='ratio = 1')
axes[2].axhline(1.5, color='red', ls='--', lw=0.9, label='overfit threshold')
axes[2].set_title('Overfitting ratio  (val / train loss)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('val / train')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

_best_ep = int(np.argmin(_val_np)) + 1
print(f'Best val loss at epoch {_best_ep}: {_val_np[_best_ep-1]:.6f}')
print(f'Final  train / val: {_train_np[-1]:.6f} / {_val_np[-1]:.6f}')
print(f'Final  overfitting ratio: {_ratio[-1]:.3f}  (>1.5 suggests overfitting)')

trained_model = model

# Training Diagnostics

The cells below help answer two key questions:

1. **Is the model overfitting?** — Compare one-step predictions on a *training* segment vs a *test* segment.  If training looks much better than test, the network has memorised the training distribution.
2. **Which output dimensions are hardest to learn?** — Per-channel loss curves show whether some velocity / angular-rate components converge more slowly or diverge on validation while continuing to improve on training.


In [ ]:
# ── Overfitting check: one-step predictions on training vs test segment ──
# A healthy model should show similar prediction quality on both splits.
# If the training segment is predicted almost perfectly while the test
# segment has large errors, the network is overfitting.

import torch, numpy as np, matplotlib.pyplot as plt

model = trained_model.to(device)
model.eval()

_VW_NAMES = ["vx", "vy", "vz", "w_roll", "w_pitch", "w_yaw"]

def _one_step_segment(seg_np):
    """Return (t_rel, gt, phys, corr) arrays (N-1, 12) for a data segment."""
    states_np, controls_np, dt_np = configure_data(seg_np)
    X  = torch.tensor(states_np[:, :12], dtype=torch.float32, device=device)
    U  = torch.tensor(controls_np,       dtype=torch.float32, device=device)
    dt = torch.tensor(dt_np.squeeze(-1), dtype=torch.float32, device=device)
    X_curr = X[:-1]; U_curr = U[:-1]; dt_curr = dt[1:].view(-1, 1)
    with torch.no_grad():
        xdot   = system_dynamics(X_curr, U_curr, mass=mass, inertia=inertia, g=g)
        X_phys = X_curr + xdot * dt_curr
        X_corr = predict_onestep_over_segment(model, X_curr, U_curr, dt_curr, mass, inertia, g)
    t_rel = seg_np[1:, 0] - seg_np[0, 0]
    return t_rel, X[1:].cpu().numpy(), X_phys.cpu().numpy(), X_corr.cpu().numpy()

# --- build a representative training segment (first ~1 s of train/val block) ---
DIAGNOSTIC_SEGMENT_DURATION = 1.0  # seconds of flight used for the overfitting check
seg_train = data_train_val[:int(DIAGNOSTIC_SEGMENT_DURATION / dt_nominal) + 1, :]
seg_test  = data_test_segments[0]

fig, axes = plt.subplots(6, 2, figsize=(16, 3 * 6), sharex=False)
fig.suptitle(
    "Overfitting check — one-step predictions\n"
    "Left: training segment  |  Right: test segment\n"
    "(similar quality on both splits = good generalisation)",
    fontsize=12)

for col, (seg, label) in enumerate([(seg_train, "Train segment"),
                                     (seg_test,  "Test segment")]):
    if seg.shape[0] < 3:
        continue
    t, gt, phys, corr = _one_step_segment(seg)
    rmse_phys = float(np.sqrt(np.mean((phys[:, 6:12] - gt[:, 6:12])**2)))
    rmse_corr = float(np.sqrt(np.mean((corr[:, 6:12] - gt[:, 6:12])**2)))
    for row, name in enumerate(_VW_NAMES):
        ax = axes[row, col]
        ax.plot(t, gt[:,   6 + row], "k-",   lw=2,   label="Ground truth")
        ax.plot(t, phys[:, 6 + row], "C0--", lw=1.4, label="Physics only")
        ax.plot(t, corr[:, 6 + row], "C1-",  lw=1.4, label="Physics + NN")
        ax.set_ylabel(name); ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(f"{label}\nRMSE phys={rmse_phys:.5f}  corr={rmse_corr:.5f}")
        if row == 5:
            ax.set_xlabel("Time [s]")
        if row == 0 and col == 0:
            ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


In [ ]:
# ── Per-channel loss curves ──
# Each panel shows the MSE for one output dimension (vx … w_yaw).
# If train loss keeps falling while val loss rises or plateaus on a
# specific channel, that channel is the main source of overfitting.

import matplotlib.pyplot as plt

CH_NAMES  = ["vx", "vy", "vz", "w_roll", "w_pitch", "w_yaw"]
epochs_ran = len(per_ch_train[0])
ep = range(epochs_ran)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ch in range(6):
    ax = axes[ch]
    ax.semilogy(ep, per_ch_train[ch], label="train", alpha=0.85)
    ax.semilogy(ep, per_ch_val[ch],   label="val",   alpha=0.85)
    ax.set_title(f"Channel: {CH_NAMES[ch]}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (log scale)")
    ax.grid(True, which='both', alpha=0.4); ax.legend()

plt.suptitle("Per-channel MSE — train vs validation", fontsize=13)
plt.tight_layout(); plt.show()


# Plotting the results

In [ ]:
@torch.no_grad()
def _f_phys_nn(model, x, u, dt, mass, inertia, g, use_nn: bool):
    """
    RHS for RK4: x_dot = f_phys(x,u) + f_nn(x,u)
    NN output is discrete Δv,Δw per step, so we convert to derivative correction by /dt.
    """
    if dt.ndim == 1:
        dt = dt.view(-1, 1)
    elif dt.ndim == 0:
        dt = dt.view(1, 1)

    xdot = system_dynamics(x, u, mass, inertia, g)  # (B,12)

    if not use_nn:
        return xdot

    delta_vw = model(x, u)  # (B,6) = [Δv, Δw] for a step of length dt
    delta_v = delta_vw[:, 0:3]
    delta_w = delta_vw[:, 3:6]

    # Avoid divide-by-zero (shouldn't happen if dt>0, but safe)
    dt_safe = torch.clamp(dt, min=1e-8)

    xdot_corr = torch.zeros_like(xdot)
    xdot_corr[:, 6:9] = delta_v / dt_safe   # v_dot correction
    xdot_corr[:, 9:12] = delta_w / dt_safe  # w_dot correction

    return xdot + xdot_corr


@torch.no_grad()
def _rk4_step(model, x, u, dt, mass, inertia, g, use_nn: bool):
    if dt.ndim == 1:
        dt = dt.view(-1, 1)
    elif dt.ndim == 0:
        dt = dt.view(1, 1)

    k1 = _f_phys_nn(model, x,               u, dt, mass, inertia, g, use_nn)
    k2 = _f_phys_nn(model, x + 0.5*dt*k1,   u, dt, mass, inertia, g, use_nn)
    k3 = _f_phys_nn(model, x + 0.5*dt*k2,   u, dt, mass, inertia, g, use_nn)
    k4 = _f_phys_nn(model, x + 1.0*dt*k3,   u, dt, mass, inertia, g, use_nn)

    return x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


@torch.no_grad()
def _euler_step(model, x, u, dt, mass, inertia, g, use_nn: bool):
    """
    Euler 'step map' version:
    - if use_nn: use your existing compute_new_pos_angles (physics + Δv,Δw)
    - else: pass Δv,Δw = 0 => pure physics through same pipeline
    """
    if dt.ndim == 1:
        dt = dt.view(-1, 1)
    elif dt.ndim == 0:
        dt = dt.view(1, 1)

    if use_nn:
        delta_vw = model(x, u)  # (B,6)
    else:
        delta_vw = torch.zeros((x.shape[0], 6), device=x.device, dtype=x.dtype)

    return compute_new_pos_angles(delta_vw, x, u, dt, mass, inertia, g)


@torch.no_grad()
def multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g, use_nn=True, integrator="euler"):
    """
    integrator: "euler" or "rk4"
    X0:     (12,)
    U_seq:  (N-1,4)
    dt_seq: (N-1,)
    returns X_hat: (N,12)
    """
    if dt_seq.ndim != 1:
        dt_seq = dt_seq.view(-1)

    N_minus_1 = U_seq.shape[0]
    dev = X0.device
    dtype = X0.dtype

    X_hat = torch.zeros((N_minus_1 + 1, 12), device=dev, dtype=dtype)
    X_hat[0] = X0

    x = X0.view(1, 12)

    integrator = integrator.lower()
    if integrator not in ("euler", "rk4"):
        raise ValueError(f"integrator must be 'euler' or 'rk4', got {integrator}")

    for k in range(N_minus_1):
        u = U_seq[k].view(1, 4)
        dt = dt_seq[k].view(1, 1)

        if integrator == "euler":
            x_next = _euler_step(model, x, u, dt, mass, inertia, g, use_nn)
        else:
            x_next = _rk4_step(model, x, u, dt, mass, inertia, g, use_nn)

        X_hat[k + 1] = x_next.view(12)
        x = x_next

    return X_hat

In [ ]:
@torch.no_grad()
def rollout_multistep_all(model, X_true, U_true, dt_true,
                          mass=2.0, inertia=torch.tensor([0.0217, 0.0217, 0.04]), g=9.81,
                          horizon_sec=1.0):
    """Roll out up to horizon_sec seconds from X_true[0]."""
    device  = next(model.parameters()).device
    X_true  = X_true.to(device)
    U_true  = U_true.to(device)
    dt_true = dt_true.to(device).view(-1)

    cum = torch.cumsum(dt_true, dim=0)
    K   = int((cum <= horizon_sec).sum().item())
    K   = max(1, min(K, X_true.shape[0] - 1))

    xk     = X_true[0:1]        # (1,12)
    X_pred = [xk.squeeze(0)]
    X_ref  = [X_true[0]]

    for k in range(K):
        uk = U_true[k:k+1]          # (1,4)
        dt = dt_true[k].view(1, 1)  # (1,1)
        xk = rollout_step_residual6(model, xk, uk, dt, mass=mass, inertia=inertia, g=g)
        X_pred.append(xk.squeeze(0))
        X_ref.append(X_true[k + 1])

    return torch.stack(X_pred, dim=0), torch.stack(X_ref, dim=0)


In [ ]:
import torch

@torch.no_grad()
def rollout_step_residual6(model, xk, uk, dt,
                           mass=2.0,
                           inertia=torch.tensor([0.0217, 0.0217, 0.04]),
                           g=9.81):
    if dt.ndim == 1:
        dt = dt.view(-1, 1)

    delta_vw = model(xk, uk)  # (B,6)
    x_next = compute_new_pos_angles(delta_vw, xk, uk, dt, mass, inertia, g)  # (B,12)
    return x_next

In [ ]:
@torch.no_grad()
def predict_onestep_over_segment(model, X_curr, U_curr, dt_curr, mass, inertia, g):
    if dt_curr.ndim == 1:
        dt_curr = dt_curr.view(-1, 1)
    return rollout_step_residual6(model, X_curr, U_curr, dt_curr, mass=mass, inertia=inertia, g=g)

@torch.no_grad()

def rollout_multistep_over_segment(model, X0, U, dt, mass, inertia, g):
    if dt.ndim == 1:
        dt = dt.view(-1, 1)

    xk = X0
    preds = []
    for k in range(U.shape[0]):
        xk = rollout_step_residual6(model, xk, U[k:k+1], dt[k:k+1], mass=mass, inertia=inertia, g=g)
        preds.append(xk)
    return torch.cat(preds, dim=0)

In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_rollout_error(X_pred, X_ref, dt=None, title="3s multistep rollout error"):

    err = (X_pred - X_ref).detach().cpu()  # (K+1,12)

    channel_names = [
        "x", "y", "z",
        "roll", "pitch", "yaw",
        "vx", "vy", "vz",
        "w_roll", "w_pitch", "w_yaw",
    ]

    # x-axis
    if dt is None:
        t = torch.arange(err.shape[0])
        xlab = "step"
    else:
        dt = torch.as_tensor(dt).detach().cpu().view(-1)
        # If dt is length K (transitions), build K+1 timestamps
        if dt.numel() == err.shape[0] - 1:
            t = torch.cat([torch.zeros(1), torch.cumsum(dt, dim=0)])
        else:
            t = torch.cumsum(dt[:err.shape[0]], dim=0)
        xlab = "time [s]"

    plt.figure(figsize=(6, 5))
    for i in range(err.shape[1]):
        plt.plot(t.numpy(), err[:, i].abs().numpy(), label=channel_names[i])

    plt.title(title)
    plt.xlabel(xlab)
    plt.ylabel("|error|")
    plt.grid(True)
    plt.legend(ncol=3, fontsize=9)  # adjust columns/font as you like
    plt.tight_layout()
    plt.show()

    rmse = torch.sqrt(torch.mean(err**2, dim=0))
    print("RMSE per channel:")
    for name, val in zip(channel_names, rmse.numpy()):
        print(f"{name:>7s}: {val:.6f}")

for seg_i, seg in enumerate(data_test_segments):
    states, controls, dt_seg = configure_data(seg)

    X_true = torch.tensor(states[:, :12], dtype=torch.float32, device=device)
    U_true = torch.tensor(controls, dtype=torch.float32, device=device)
    dt_true = torch.tensor(dt_seg.squeeze(), dtype=torch.float32, device=device)

    # Your existing function (whatever it returns)
    X_pred, X_ref = rollout_multistep_all(model, X_true, U_true, dt_true)

    plot_rollout_error(
        X_pred, X_ref,
        title=f"Test segment {seg_i} - 3s multistep rollout (t ∈ [{float(seg[0,0]):.2f}, {float(seg[-1,0]):.2f}] s)"
    )

In [ ]:
@torch.no_grad()
def predict_next_state_residual6(model, X_curr, U_curr, dt,
                                mass=2.0,
                                inertia=torch.tensor([0.0217, 0.0217, 0.04]),
                                g=9.81):
    """
    X_curr: (B,12)
    U_curr: (B,4)
    dt:     (B,) or (B,1)
    returns X_next_pred: (B,12)
    """
    if dt.ndim == 1:
        dt = dt.view(-1, 1)

    # NN correction (B,6)
    delta_vw = model(X_curr, U_curr)

    # Use your existing integrator that combines physics + delta_vw
    X_next_pred = compute_new_pos_angles(delta_vw, X_curr, U_curr, dt, mass, inertia, g)
    return X_next_pred

**Step 7: Testing**

# One step (euler, xavier, ELu)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

STATE_NAMES = ["x","y","z","roll","pitch","yaw","vx","vy","vz","w_roll","w_pitch","w_yaw"]

@torch.no_grad()
def one_step_predictions(model, X_curr, U_curr, dt_curr, mass, inertia, g):
    """
    X_curr: (N-1,12)
    U_curr: (N-1,4)
    dt_curr: (N-1,1) or (N-1,)
    returns:
      X_next_phys: (N-1,12)
      X_next_corr: (N-1,12)
    """
    if dt_curr.ndim == 1:
        dt_curr = dt_curr.view(-1, 1)

    # physics-only
    xdot = system_dynamics(X_curr, U_curr, mass=mass, inertia=inertia, g=g)  # (N-1,12)
    X_next_phys = X_curr + xdot * dt_curr

    # corrected
    delta_vw = model(X_curr, U_curr)  # (N-1,6)
    X_next_corr = compute_new_pos_angles(delta_vw, X_curr, U_curr, dt_curr, mass, inertia, g)

    return X_next_phys, X_next_corr

def plot_two_column_next_state(t_next, X_next_gt, X_next_phys, X_next_corr, title):
    """
    t_next: (N-1,)
    X_next_*: (N-1,12)
    """
    n_states = 12
    fig, axes = plt.subplots(n_states, 2, figsize=(16, 2.2 * n_states), sharex=True)

    for i in range(n_states):
        axL = axes[i, 0]
        axL.plot(t_next, X_next_gt[:, i], "k-", lw=2, label="ground truth (next)")
        axL.plot(t_next, X_next_phys[:, i], "C0--", lw=1.6, label="physics (1-step)")
        axL.set_ylabel(STATE_NAMES[i])
        axL.grid(True, alpha=0.3)
        if i == 0:
            axL.set_title("Physics (1-step) vs Ground Truth")
        if i == n_states - 1:
            axL.set_xlabel("time [s] (next sample)")

        axR = axes[i, 1]
        axR.plot(t_next, X_next_gt[:, i], "k-", lw=2, label="ground truth (next)")
        axR.plot(t_next, X_next_corr[:, i], "C1--", lw=1.6, label="physics + NN (1-step)")
        axR.grid(True, alpha=0.3)
        if i == 0:
            axR.set_title("Corrected (1-step) vs Ground Truth")
        if i == n_states - 1:
            axR.set_xlabel("time [s] (next sample)")

    handlesL, labelsL = axes[0, 0].get_legend_handles_labels()
    handlesR, labelsR = axes[0, 1].get_legend_handles_labels()
    fig.legend(handlesL + handlesR, labelsL + labelsR, loc="upper left", ncol=4, frameon=True)

    fig.suptitle(title, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()

# ---------- run on your test segments ----------
mass = 2.0
inertia = torch.tensor([0.0217, 0.0217, 0.04], dtype=torch.float32, device=device)
g = 9.81

model = trained_model.to(device)
model.eval()

for seg_i, seg in enumerate(data_test_segments):
    # seg: (N, dataset_cols)
    states_np, controls_np, dt_np = configure_data(seg)  # states_np: (N,18), controls_np: (N,4), dt_np: (N,1)

    X = torch.tensor(states_np[:, :12], dtype=torch.float32, device=device)  # (N,12)
    U = torch.tensor(controls_np, dtype=torch.float32, device=device)        # (N,4)
    dt = torch.tensor(dt_np.squeeze(-1), dtype=torch.float32, device=device) # (N,)

    # Build 1-step pairs inside the segment
    X_curr = X[:-1, :]
    X_next_gt = X[1:, :]

    U_curr = U[:-1, :]

    # dt that takes you from curr -> next (dt[0] is often 0 due to diff; so use dt[1:])
    dt_curr = dt[1:]  # length N-1

    # time axis: next sample times
    t_next = seg[1:, 0]

    X_next_phys, X_next_corr = one_step_predictions(
        model=model,
        X_curr=X_curr,
        U_curr=U_curr,
        dt_curr=dt_curr,
        mass=mass,
        inertia=inertia,
        g=g
    )

    plot_two_column_next_state(
        t_next=t_next,
        X_next_gt=X_next_gt.detach().cpu().numpy(),
        X_next_phys=X_next_phys.detach().cpu().numpy(),
        X_next_corr=X_next_corr.detach().cpu().numpy(),
        title=f"One-step test | segment {seg_i} | t ∈ [{float(seg[0,0]):.2f}, {float(seg[-1,0]):.2f}] s"
    )

# Multistep (Euler, xavier, ELu)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

STATE_NAMES = ["x","y","z","roll","pitch","yaw","vx","vy","vz","w_roll","w_pitch","w_yaw"]

# multistep_rollout (Euler + RK4) is defined in the integrator cell above.

def plot_two_column_next_state(t_next, X_next_gt, X_next_phys, X_next_corr, title):
    """
    Two-column plot: left = physics-only, right = physics + NN correction.
    t_next     : 1-D array of time values (seconds from segment start)
    X_next_*   : (N-1, 12) numpy arrays
    """
    n_states = 12
    fig, axes = plt.subplots(n_states, 2, figsize=(16, 2.2 * n_states), sharex=True)

    for i in range(n_states):
        axL = axes[i, 0]
        axL.plot(t_next, X_next_gt[:,   i], "k-",  lw=2,   label="ground truth")
        axL.plot(t_next, X_next_phys[:, i], "C0--", lw=1.6, label="physics only")
        axL.set_ylabel(STATE_NAMES[i]); axL.grid(True, alpha=0.3)
        if i == 0:          axL.set_title("Physics only vs Ground Truth")
        if i == n_states-1: axL.set_xlabel("time [s] from segment start")

        axR = axes[i, 1]
        axR.plot(t_next, X_next_gt[:,   i], "k-",  lw=2,   label="ground truth")
        axR.plot(t_next, X_next_corr[:, i], "C1--", lw=1.6, label="physics + NN")
        axR.grid(True, alpha=0.3)
        if i == 0:          axR.set_title("Physics + NN vs Ground Truth")
        if i == n_states-1: axR.set_xlabel("time [s] from segment start")

    hL, lL = axes[0,0].get_legend_handles_labels()
    hR, lR = axes[0,1].get_legend_handles_labels()
    fig.legend(hL + hR, lL + lR, loc="upper left", ncol=4, frameon=True)
    fig.suptitle(title, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()


# ---------- run on test segments (Euler multi-step) ----------
mass    = 2.0
inertia = torch.tensor([0.0217, 0.0217, 0.04], dtype=torch.float32, device=device)
g       = 9.81

model = trained_model.to(device)
model.eval()

for seg_i, seg in enumerate(data_test_segments):
    states_np, controls_np, dt_np = configure_data(seg)

    X  = torch.tensor(states_np[:, :12], dtype=torch.float32, device=device)
    U  = torch.tensor(controls_np,       dtype=torch.float32, device=device)
    dt = torch.tensor(dt_np.squeeze(-1), dtype=torch.float32, device=device)

    X_next_gt = X[1:, :]   # ground-truth "next" states
    # Relative time axis (0 → ~1 s)
    t_next = seg[1:, 0] - seg[0, 0]

    X0     = X[0]           # (12,)
    U_seq  = U[:-1, :]      # (N-1, 4)
    dt_seq = dt[1:]         # (N-1,)

    X_hat_phys = multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g,
                                   use_nn=False, integrator="euler")  # (N,12)
    X_hat_corr = multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g,
                                   use_nn=True,  integrator="euler")  # (N,12)

    plot_two_column_next_state(
        t_next=t_next,
        X_next_gt=X_next_gt.detach().cpu().numpy(),
        X_next_phys=X_hat_phys[1:].detach().cpu().numpy(),
        X_next_corr=X_hat_corr[1:].detach().cpu().numpy(),
        title=f"Multi-step (Euler) | segment {seg_i} | t₀={float(seg[0,0]):.2f} s")


**Step 8: Plot position error in 3D**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

model = trained_model.to(device)
model.eval()

for seg_idx, seg in enumerate(data_test_segments, start=1):
    if seg.shape[0] < 3:
        continue

    states, controls, dt_seg = configure_data(seg)

    X = torch.tensor(states[:, :12], dtype=torch.float32, device=device)           # (N,12)
    U = torch.tensor(controls, dtype=torch.float32, device=device)                 # (N,4)
    dt = torch.tensor(dt_seg.squeeze(-1), dtype=torch.float32, device=device)      # (N,)

    # Build curr->next pairs
    X_curr = X[:-1]          # (N-1,12)
    X_next = X[1:]           # (N-1,12)
    U_curr = U[:-1]          # (N-1,4)
    dt_curr = dt[1:]         # (N-1,)  (skip dt[0] which is often 0)

    with torch.no_grad():
        # One-step (teacher forcing): predict next from true current
        X_pred_onestep = predict_onestep_over_segment(model, X_curr, U_curr, dt_curr, mass, inertia, g)  # (N-1,12)

        # Multi-step (recursive): predict next from previous prediction
        X_pred_multistep = rollout_multistep_over_segment(model, X[0:1], U_curr, dt_curr, mass, inertia, g)  # (N-1,12)

    # Ground-truth positions (N points)
    gt_pos = X[:, 0:3].detach().cpu().numpy()

    # Predicted position trajectories (N points)
    onestep_pos = np.vstack([X[0, 0:3].detach().cpu().numpy(),
                             X_pred_onestep[:, 0:3].detach().cpu().numpy()])

    multistep_pos = np.vstack([X[0, 0:3].detach().cpu().numpy(),
                               X_pred_multistep[:, 0:3].detach().cpu().numpy()])

    # Position errors for next-state (N-1 points)
    err_onestep = torch.linalg.norm(X_pred_onestep[:, 0:3] - X_next[:, 0:3], dim=1).detach().cpu().numpy()
    err_multistep = torch.linalg.norm(X_pred_multistep[:, 0:3] - X_next[:, 0:3], dim=1).detach().cpu().numpy()

    rmse_onestep = float(torch.sqrt(torch.mean((X_pred_onestep[:, 0:3] - X_next[:, 0:3]) ** 2)).item())
    rmse_multistep = float(torch.sqrt(torch.mean((X_pred_multistep[:, 0:3] - X_next[:, 0:3]) ** 2)).item())

    # -------- Plot --------
    fig = plt.figure(figsize=(14, 5))

    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    ax1.plot(gt_pos[:, 0], gt_pos[:, 1], gt_pos[:, 2], label="Ground truth", linewidth=2)
    ax1.plot(onestep_pos[:, 0], onestep_pos[:, 1], onestep_pos[:, 2], "--", label="One-step", linewidth=2)
    ax1.plot(multistep_pos[:, 0], multistep_pos[:, 1], multistep_pos[:, 2], ":", label="Multi-step", linewidth=2)
    ax1.set_title(f"Segment {seg_idx}: 3D position trajectory")
    ax1.set_xlabel("x [m]")
    ax1.set_ylabel("y [m]")
    ax1.set_zlabel("z [m]")
    ax1.legend()

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot(err_onestep, label=f"One-step error (RMSE={rmse_onestep:.4f})")
    ax2.plot(err_multistep, label=f"Multi-step error (RMSE={rmse_multistep:.4f})")
    ax2.set_title(f"Segment {seg_idx}: position error norm")
    ax2.set_xlabel("Sample index")
    ax2.set_ylabel("||p_pred - p_true|| [m]")
    ax2.grid(True)
    ax2.legend()

    plt.tight_layout()
    plt.show()

    print(f"Segment {seg_idx} position RMSE -> one-step: {rmse_onestep:.6f}, multi-step: {rmse_multistep:.6f}")

# Euler and RK4 Multistep Comparison

In [ ]:
# ---- Multi-step comparison: Euler vs RK4, physics-only vs physics+NN ----
# Plots show 1 second of trajectory (relative time 0 → ~1 s).

mass    = 2.0
inertia = torch.tensor([0.0217, 0.0217, 0.04], dtype=torch.float32, device=device)
g       = 9.81

model = trained_model.to(device)
model.eval()

for seg_i, seg in enumerate(data_test_segments):
    states_np, controls_np, dt_np = configure_data(seg)

    X  = torch.tensor(states_np[:, :12], dtype=torch.float32, device=device)
    U  = torch.tensor(controls_np,       dtype=torch.float32, device=device)
    dt = torch.tensor(dt_np.squeeze(-1), dtype=torch.float32, device=device)

    X_next_gt = X[1:, :]
    # Relative time axis starting from 0
    t_next = seg[1:, 0] - seg[0, 0]

    X0     = X[0]
    U_seq  = U[:-1, :]
    dt_seq = dt[1:]

    # --- Euler ---
    X_hat_phys_e = multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g,
                                     use_nn=False, integrator="euler")
    X_hat_corr_e = multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g,
                                     use_nn=True,  integrator="euler")

    # --- RK4 ---
    X_hat_phys_r = multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g,
                                     use_nn=False, integrator="rk4")
    X_hat_corr_r = multistep_rollout(model, X0, U_seq, dt_seq, mass, inertia, g,
                                     use_nn=True,  integrator="rk4")

    plot_two_column_next_state(
        t_next=t_next,
        X_next_gt=X_next_gt.detach().cpu().numpy(),
        X_next_phys=X_hat_phys_e[1:].detach().cpu().numpy(),
        X_next_corr=X_hat_corr_e[1:].detach().cpu().numpy(),
        title=f"Multi-step (Euler) | segment {seg_i} | t₀={float(seg[0,0]):.2f} s")

    plot_two_column_next_state(
        t_next=t_next,
        X_next_gt=X_next_gt.detach().cpu().numpy(),
        X_next_phys=X_hat_phys_r[1:].detach().cpu().numpy(),
        X_next_corr=X_hat_corr_r[1:].detach().cpu().numpy(),
        title=f"Multi-step (RK4)   | segment {seg_i} | t₀={float(seg[0,0]):.2f} s")


# NN Internals Analysis

These cells reveal *how* the network is doing its job:

* **Correction magnitude vs physics** — If the NN corrections are much larger than the physics step, the NN is compensating for a poorly calibrated physical model rather than learning small residual effects.  This suggests improving the physical parameters or using a more expressive integrator.
* **Weight / bias distributions** — Healthy networks have weights roughly normally distributed around zero.  Very large weights indicate overfitting or missing regularisation.  Dead neurons (weights collapsed to ~0) can appear with ReLU but are less common with ELU.


In [ ]:
# ── NN correction magnitude vs physics-step contribution ──
# Histograms compare Δv / Δw produced by the NN to the velocity change
# due to physics alone over one time-step.  The ratio tells us how much
# work the NN is doing relative to the physics model.

import torch, numpy as np, matplotlib.pyplot as plt

model = trained_model.to(device)
model.eval()

all_delta_v, all_delta_w = [], []
all_phys_v,  all_phys_w  = [], []

for seg in data_test_segments:
    if seg.shape[0] < 3:
        continue
    states_np, controls_np, dt_np = configure_data(seg)
    X  = torch.tensor(states_np[:, :12], dtype=torch.float32, device=device)
    U  = torch.tensor(controls_np,       dtype=torch.float32, device=device)
    dt = torch.tensor(dt_np.squeeze(-1), dtype=torch.float32, device=device).view(-1, 1)
    X_curr = X[:-1]; U_curr = U[:-1]; dt_c = dt[1:]
    with torch.no_grad():
        delta_vw = model(X_curr, U_curr)
        xdot     = system_dynamics(X_curr, U_curr, mass=mass, inertia=inertia, g=g)
    all_delta_v.append(delta_vw[:, 0:3].cpu())
    all_delta_w.append(delta_vw[:, 3:6].cpu())
    all_phys_v.append((xdot[:, 6:9]  * dt_c).cpu())
    all_phys_w.append((xdot[:, 9:12] * dt_c).cpu())

delta_v = torch.cat(all_delta_v).numpy()
delta_w = torch.cat(all_delta_w).numpy()
phys_v  = torch.cat(all_phys_v).numpy()
phys_w  = torch.cat(all_phys_w).numpy()

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
v_names = ["vx", "vy", "vz"]
w_names = ["w_roll", "w_pitch", "w_yaw"]

for i, name in enumerate(v_names):
    ax = axes[0, i]
    ax.hist(phys_v[:, i],  bins=60, alpha=0.6, label="physics Δv", color="C0")
    ax.hist(delta_v[:, i], bins=60, alpha=0.6, label="NN correction Δv", color="C1")
    ax.set_title(f"{name}"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

for i, name in enumerate(w_names):
    ax = axes[1, i]
    ax.hist(phys_w[:, i],  bins=60, alpha=0.6, label="physics Δw", color="C0")
    ax.hist(delta_w[:, i], bins=60, alpha=0.6, label="NN correction Δw", color="C1")
    ax.set_title(f"{name}"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(
    "Distribution of NN corrections vs physics-step contributions\n"
    "(wide NN distribution relative to physics → NN compensating large model errors)",
    fontsize=11)
plt.tight_layout(); plt.show()

_EPS = 1e-9  # small constant to avoid division by zero in RMS ratio
print("=== Correction RMS vs Physics-step RMS ===")
for i, name in enumerate(v_names):
    r_nn = float(np.sqrt(np.mean(delta_v[:, i]**2)))
    r_ph = float(np.sqrt(np.mean(phys_v[:, i]**2)))
    print(f"  {name}: NN={r_nn:.5f}  Physics={r_ph:.5f}  ratio={r_nn/max(r_ph,_EPS):.3f}")
for i, name in enumerate(w_names):
    r_nn = float(np.sqrt(np.mean(delta_w[:, i]**2)))
    r_ph = float(np.sqrt(np.mean(phys_w[:, i]**2)))
    print(f"  {name}: NN={r_nn:.5f}  Physics={r_ph:.5f}  ratio={r_nn/max(r_ph,_EPS):.3f}")


In [ ]:
# ── Weight and bias distributions ──
# Healthy: symmetric, roughly normal distribution centred near zero.
# Warning signs:
#   - Very large weights (|w| >> 1) after L2 regularisation → still overfitting.
#   - Most weights collapsing to zero → network capacity is unused.
#   - Strong asymmetry or bimodal distribution → training instability.

import matplotlib.pyplot as plt, numpy as np

model = trained_model

all_weights, all_biases = [], []
for pname, param in model.named_parameters():
    data = param.detach().cpu().numpy().flatten()
    if "weight" in pname:
        all_weights.append(data)
    elif "bias" in pname:
        all_biases.append(data)

w_flat = np.concatenate(all_weights) if all_weights else np.array([])
b_flat = np.concatenate(all_biases)  if all_biases  else np.array([])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(w_flat, bins=80, color="steelblue", alpha=0.85)
axes[0].axvline(0, color='k', lw=1.5, ls='--')
axes[0].set_title("Weight distribution (all layers)")
axes[0].set_xlabel("Weight value"); axes[0].set_ylabel("Count")
axes[0].grid(True, alpha=0.3)

axes[1].hist(b_flat, bins=40, color="darkorange", alpha=0.85)
axes[1].axvline(0, color='k', lw=1.5, ls='--')
axes[1].set_title("Bias distribution (all layers)")
axes[1].set_xlabel("Bias value"); axes[1].set_ylabel("Count")
axes[1].grid(True, alpha=0.3)

total_params = sum(p.numel() for p in model.parameters())
plt.suptitle(f"Network weight/bias distributions  (total trainable params: {total_params:,})",
             fontsize=12)
plt.tight_layout(); plt.show()

print("=== Layer-wise weight L2 norms ===")
for pname, param in model.named_parameters():
    if 'weight' in pname:
        l2 = float(param.detach().norm(2).item())
        print(f"  {pname}: L2 = {l2:.4f}")
